# GCI In-Class Competition: Home Credit Default Risk
## Notebook 3 v2: Ensemble Training (evidence-based rebuild)

**What changed vs the old NB3 (and why)** — every change below was verified by local benchmarks
(full 171,202-row 5-fold CV, identical folds):
1. **No hold-out set.** The old 10% hold-out wasted data and its small size (6.1k rows) inflated
   AUC by ~+0.006. Selecting on full OOF is both safer and worth **+0.0028**.
2. **Feature recipe (validated):** original 101 engineered features + 17 new
   (EXT×AGE, NaN-safe EXT products, burden ratios, NAN_COUNT) − 18 dead features.
   Combo-categorical TE was tested and **rejected** (hurt −0.0007: sparse cells on small data).
3. **Member list (validated):** LightGBM (tuned params, 3 seeds), XGBoost (tuned), CatBoost
   (tuned), sklearn HistGradientBoosting. LogReg/RF/DART tested and rejected (too weak or diluting).
4. **Strategy selected on OOF by hill-climb + 4-way grid** — never on a hold-out.
5. **No "retrain on full data" cell.** The old cell-12 retrained XGBoost with LightGBM's
   iteration count and CatBoost with a fixed 1000 iters, then overwrote the calibrated blend
   with a squashed distribution (verified: std 0.099 vs 0.213). Final submission = CV-averaged
   test predictions, always.

**Reference numbers (local, same folds, v3 recipe):**
LGBnew3 0.76155 · XGBnew3 0.76171 · XGBold3 0.76167 · CAT2 0.76028 · HGB 0.75588
final v3 blend → **CV 0.76224** (vs old pipeline true OOF 0.7587 → +0.0035)

In [ ]:
# 1. Setup
!pip install catboost -q
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
import time, os, gc, json, joblib
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5
N_ESTIMATORS = 3000
EARLY_STOPPING = 200
print("LightGBM:", lgb.__version__, "| XGBoost:", xgb.__version__, "| CatBoost:", cb.__version__)

In [ ]:
# 2. Mount Drive & load engineered data + tuned params
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/GCI Competition'

train = pd.read_parquet(f'{WORK_DIR}/features/train_engineered.parquet')
test  = pd.read_parquet(f'{WORK_DIR}/features/test_engineered.parquet')
with open(f'{WORK_DIR}/features/feature_cols.json') as f:
    feature_cols = json.load(f)
y = np.load(f'{WORK_DIR}/features/y_train.npy')
with open(f'{WORK_DIR}/optuna_studies/all_best_params.json') as f:
    BEST = json.load(f)
test_ids = test['SK_ID_CURR'].values

X0 = train[feature_cols].fillna(-999)
X_test0 = test[feature_cols].fillna(-999)
print(f"train {X0.shape} | test {X_test0.shape} | positive rate {y.mean():.4f}")

# --- OVERRIDE with re-tuned params (Optuna on the final engineered feature set, verified locally) ---
BEST['lightgbm'] = {'learning_rate': 0.02216201875371203, 'num_leaves': 27, 'max_depth': 4,
    'min_child_samples': 140, 'subsample': 0.7323592099410596, 'subsample_freq': 1,
    'colsample_bytree': 0.5317791751430119, 'reg_alpha': 0.008569331925053986,
    'reg_lambda': 0.009452571391072306, 'min_split_gain': 0.7296061783380641,
    'min_child_weight': 0.08178476574339538}
BEST['xgboost'] = {'learning_rate': 0.023002713466747066, 'max_depth': 3,
    'min_child_weight': 18.30736186227178, 'subsample': 0.7632235887667118,
    'colsample_bytree': 0.4649614716389921, 'colsample_bylevel': 0.7905425221921015,
    'reg_alpha': 0.0008744530679082095, 'reg_lambda': 9.37721343436773,
    'gamma': 0.5663803097877049}
print('params overridden with re-tuned values')


In [ ]:
# 3. Feature recipe (validated): enrich + drop dead features
DEAD = [
    'FLAG_MOBIL','ORGANIZATION_TYPE','EXT_SOURCE_2_WAS_MISSING','CNT_FAM_MEMBERS_WAS_MISSING',
    'AMT_REQ_CREDIT_BUREAU_MON_WAS_MISSING','AMT_REQ_CREDIT_BUREAU_YEAR_WAS_MISSING',
    'EXT_SOURCE_1_WAS_MISSING','EXT_SOURCE_3_WAS_MISSING','FLAG_EMP_PHONE',
    'EXT_SOURCE_3_QBIN','OCCUPATION_TYPE_FREQ','NAME_HOUSING_TYPE_FREQ','NAME_FAMILY_STATUS_FREQ',
    'NAME_EDUCATION_TYPE_FREQ','NAME_INCOME_TYPE_FREQ','CODE_GENDER_FREQ','NAME_CONTRACT_TYPE_FREQ',
    'ORGANIZATION_TYPE_FREQ',
]

def enrich(df, base_cols):
    X = df.copy()
    eps = 1e-5
    ext = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    filled = X[ext].replace(-999, np.nan)
    f = filled.fillna(filled.mean())
    age_years = (X['DAYS_BIRTH'].replace(-999, np.nan).abs() / 365.0).fillna(
        X['DAYS_BIRTH'].replace(-999, np.nan).abs().mean() / 365.0)
    for c in ext:
        X[f'{c}X_AGE'] = f[c] * age_years
    X['EXT_MEAN_X_AGE'] = f.mean(axis=1) * age_years
    X['EXT_2x3'] = f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_1x2x3_NANSAFE'] = f['EXT_SOURCE_1'] * f['EXT_SOURCE_2'] * f['EXT_SOURCE_3']
    X['EXT_2_SQ'] = f['EXT_SOURCE_2'] ** 2
    X['EXT_3_SQ'] = f['EXT_SOURCE_3'] ** 2
    X['EXT_SUM'] = f.sum(axis=1)
    emp_years = X['DAYS_EMPLOYED'].replace(-999, np.nan).abs() / 365.0
    X['CREDIT_PER_INCOME_YEAR'] = X['AMT_CREDIT'] / ((X['AMT_INCOME_TOTAL'] + eps) * emp_years.fillna(emp_years.mean()) + eps)
    X['ANNUITY_PER_FAM'] = X['AMT_ANNUITY'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['CREDIT_PER_FAM'] = X['AMT_CREDIT'] / (X['CNT_FAM_MEMBERS'].replace(0, 1) + eps)
    X['INCOME_PER_CHILD'] = X['AMT_INCOME_TOTAL'] / (X['CNT_CHILDREN'].clip(lower=0) + 1)
    X['NAN_COUNT'] = (df[base_cols] == -999).sum(axis=1).astype(np.int16)
    X['REGION_RATING_X_EXT_MEAN'] = X['REGION_RATING_CLIENT'] * f.mean(axis=1)
    return X

X = enrich(X0, feature_cols).drop(columns=[c for c in DEAD if c in X0.columns])
X_test = enrich(X_test0, feature_cols).drop(columns=[c for c in DEAD if c in X_test0.columns])
assert list(X.columns) == list(X_test.columns)
print(f"final feature frame: {X.shape[1]} columns")

In [ ]:
# 4. CV folds + fold-safe target encoding (single-col TE only; combo TE rejected by test)
with open(f'{WORK_DIR}/features/high_card_cols.json') as f:
    high_card_cols = json.load(f)
TE_COLS = [c for c in high_card_cols if c in X.columns]
CATS = ['NAME_CONTRACT_TYPE','CODE_GENDER','NAME_INCOME_TYPE','NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS','NAME_HOUSING_TYPE','OCCUPATION_TYPE','FLAG_WORK_PHONE']

def add_te(X_tr, X_va, X_te, y_tr, te_cols=TE_COLS, smoothing=10):
    X_tr, X_va, X_te = X_tr.copy(), X_va.copy(), X_te.copy()
    gs = pd.Series(y_tr, index=X_tr.index)
    gm = gs.mean()
    for col in te_cols:
        means = gs.groupby(X_tr[col]).mean()
        counts = X_tr[col].value_counts()
        sm = (means * counts + gm * smoothing) / (counts + smoothing)
        X_tr[f'{col}_TE'] = X_tr[col].map(sm).fillna(gm)
        X_va[f'{col}_TE'] = X_va[col].map(sm).fillna(gm)
        X_te[f'{col}_TE'] = X_te[col].map(sm).fillna(gm)
    return X_tr, X_va, X_te

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))
def cat_idx(Xc):
    return [list(Xc.columns).index(c) for c in CATS if c in Xc.columns]
print(f"folds ready | TE cols: {TE_COLS}")

In [ ]:
# 5. LightGBM — tuned params, 3-seed average (CPU; pip wheel has no GPU)
lgb_params = dict(BEST['lightgbm'])
oof_lgb = np.zeros(len(X)); test_lgb = np.zeros(len(X_test))
t0 = time.time()
for seed in [42, 1337, 2026]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        m = lgb.LGBMClassifier(**lgb_params, n_estimators=N_ESTIMATORS,
                               random_state=seed, verbose=-1, n_jobs=2)
        m.fit(X_tr, y[tri], eval_set=[(X_va, y[vai])], eval_metric='auc',
              categorical_feature=cat_idx(X_tr),
              callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])
        oof_s[vai] = m.predict_proba(X_va)[:, 1]
        test_s += m.predict_proba(X_te)[:, 1] / N_SPLITS
    print(f"  LGB seed {seed}: OOF AUC = {roc_auc_score(y, oof_s):.5f}")
    oof_lgb += oof_s / 3
    test_lgb += test_s / 3
auc_lgb = roc_auc_score(y, oof_lgb)
print(f"LGB (3-seed avg): {auc_lgb:.5f}  ({time.time()-t0:.0f}s)")
np.save(f'{WORK_DIR}/models/v2_oof_lgb.npy', oof_lgb)
np.save(f'{WORK_DIR}/models/v2_test_lgb.npy', test_lgb)

# --- OVERRIDE with re-tuned params (Optuna on the final engineered feature set, verified locally) ---
BEST['lightgbm'] = {'learning_rate': 0.02216201875371203, 'num_leaves': 27, 'max_depth': 4,
    'min_child_samples': 140, 'subsample': 0.7323592099410596, 'subsample_freq': 1,
    'colsample_bytree': 0.5317791751430119, 'reg_alpha': 0.008569331925053986,
    'reg_lambda': 0.009452571391072306, 'min_split_gain': 0.7296061783380641,
    'min_child_weight': 0.08178476574339538}
BEST['xgboost'] = {'learning_rate': 0.023002713466747066, 'max_depth': 3,
    'min_child_weight': 18.30736186227178, 'subsample': 0.7632235887667118,
    'colsample_bytree': 0.4649614716389921, 'colsample_bylevel': 0.7905425221921015,
    'reg_alpha': 0.0008744530679082095, 'reg_lambda': 9.37721343436773,
    'gamma': 0.5663803097877049}
print('params overridden with re-tuned values')


In [ ]:
# 6. XGBoost - tuned params, 3 seeds (GPU detected once, CPU fallback)
xp0 = dict(BEST['xgboost']); xp0.update({'objective':'binary:logistic','eval_metric':'auc','n_jobs':2,'verbosity':0})
use_cuda = False
try:
    _probe = xgb.DMatrix(X.iloc[:200])
    xgb.train({**xp0, 'device':'cuda', 'random_state':SEED}, _probe, num_boost_round=1)
    use_cuda = True
except Exception:
    pass
xp0 = {**xp0, 'device':'cuda'} if use_cuda else {**xp0, 'tree_method':'hist'}
print('XGB device:', 'cuda' if use_cuda else 'hist (CPU)')

oof_xgb = np.zeros(len(X)); test_xgb = np.zeros(len(X_test))
t0 = time.time()
for seed in [42, 1337, 2026]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        dtr = xgb.DMatrix(X_tr, label=y[tri]); dva = xgb.DMatrix(X_va, label=y[vai]); dte = xgb.DMatrix(X_te)
        m = xgb.train({**xp0, 'random_state':seed}, dtr, num_boost_round=N_ESTIMATORS,
                      evals=[(dva,'val')], early_stopping_rounds=EARLY_STOPPING, verbose_eval=False)
        oof_s[vai] = m.predict(dva, iteration_range=(0, m.best_iteration + 1))
        test_s += m.predict(dte, iteration_range=(0, m.best_iteration + 1)) / N_SPLITS
    print(f'  XGB seed {seed}: OOF AUC = {roc_auc_score(y, oof_s):.5f}')
    oof_xgb += oof_s / 3
    test_xgb += test_s / 3
auc_xgb = roc_auc_score(y, oof_xgb)
print(f'XGB (3-seed avg): {auc_xgb:.5f}  ({time.time()-t0:.0f}s)')
np.save(f'{WORK_DIR}/models/v2_oof_xgb.npy', oof_xgb)
np.save(f'{WORK_DIR}/models/v2_test_xgb.npy', test_xgb)

In [ ]:
# 7. CatBoost - tuned params, 2 seeds (CPU only!)
# IMPORTANT: do NOT use GPU for CAT. AUC early-stopping is not implemented for GPU
# (verified: GPU run stops on Logloss -> OOF 0.75847 vs CPU/AUC-stop 0.76033).
cp0 = dict(BEST['catboost']); cp0.update({'loss_function':'Logloss','eval_metric':'AUC','verbose':False,'allow_writing_files':False,'thread_count':2})
oof_cat = np.zeros(len(X)); test_cat = np.zeros(len(X_test))
t0 = time.time()
for seed in [42, 1337]:
    oof_s = np.zeros(len(X)); test_s = np.zeros(len(X_test))
    for fold, (tri, vai) in enumerate(FOLDS):
        X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
        ci = cat_idx(X_tr)
        m = cb.CatBoostClassifier(**cp0, iterations=N_ESTIMATORS, random_seed=seed)
        m.fit(X_tr, y[tri], eval_set=(X_va, y[vai]), cat_features=ci, early_stopping_rounds=EARLY_STOPPING)
        oof_s[vai] = m.predict_proba(X_va)[:, 1]
        test_s += m.predict_proba(X_te)[:, 1] / N_SPLITS
    print(f'  CAT seed {seed}: OOF AUC = {roc_auc_score(y, oof_s):.5f}')
    oof_cat += oof_s / 2
    test_cat += test_s / 2
auc_cat = roc_auc_score(y, oof_cat)
print(f'CAT (2-seed avg): {auc_cat:.5f}  ({time.time()-t0:.0f}s)')
np.save(f'{WORK_DIR}/models/v2_oof_cat.npy', oof_cat)
np.save(f'{WORK_DIR}/models/v2_test_cat.npy', test_cat)

In [ ]:
# 8. HistGradientBoosting — diversity member (sklearn, CPU)
oof_hgb = np.zeros(len(X)); test_hgb = np.zeros(len(X_test))
t0 = time.time()
for fold, (tri, vai) in enumerate(FOLDS):
    X_tr, X_va, X_te = add_te(X.iloc[tri], X.iloc[vai], X_test, y[tri])
    m = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=600, max_leaf_nodes=31,
                                       min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
    m.fit(X_tr, y[tri])
    oof_hgb[vai] = m.predict_proba(X_va)[:, 1]
    test_hgb += m.predict_proba(X_te)[:, 1] / N_SPLITS
auc_hgb = roc_auc_score(y, oof_hgb)
print(f"HGB: {auc_hgb:.5f}  ({time.time()-t0:.0f}s)")
np.save(f'{WORK_DIR}/models/v2_oof_hgb.npy', oof_hgb)
np.save(f'{WORK_DIR}/models/v2_test_hgb.npy', test_hgb)

In [ ]:
# 9. Strategy selection on FULL OOF: greedy hill-climb + 4-way grid
MEMBERS = {'LGB3': (oof_lgb, test_lgb), 'HGB': (oof_hgb, test_hgb),
           'XGB': (oof_xgb, test_xgb), 'CAT': (oof_cat, test_cat)}

print("=== member summary (full-data OOF) ===")
for k, (o, _) in MEMBERS.items():
    print(f"  {k}: {roc_auc_score(y, o):.5f}")

# greedy with replacement: blend = mean of chosen members (repeats allowed)
current, chosen, counts = None, [], {k: 0 for k in MEMBERS}
for step in range(10):
    best_k, best_a = None, -1.0
    for k, (o, _) in MEMBERS.items():
        cand = o.copy() if current is None else (current + o) / 2
        a = roc_auc_score(y, cand)
        if a > best_a:
            best_k, best_a = k, a
    if current is not None and best_a <= roc_auc_score(y, current):
        break
    current = MEMBERS[best_k][0].copy() if current is None else (current + MEMBERS[best_k][0]) / 2
    counts[best_k] += 1
    chosen.append(best_k)
    print(f"  hill-climb step {step+1}: +{best_k} -> {best_a:.5f} | counts { {k:v for k,v in counts.items() if v} }")

# 4-way weight grid (step 0.05) as the precise final recipe
names = list(MEMBERS.keys())
OOF_M = np.column_stack([MEMBERS[k][0] for k in names])
best = (-1.0, None)
for w1 in np.arange(0, 1.0001, 0.05):
    for w2 in np.arange(0, 1.0001 - w1, 0.05):
        for w3 in np.arange(0, 1.0001 - w1 - w2 + 1e-9, 0.05):
            w4 = 1 - w1 - w2 - w3
            if w4 < -1e-9: continue
            a = roc_auc_score(y, w1*OOF_M[:,0] + w2*OOF_M[:,1] + w3*OOF_M[:,2] + w4*OOF_M[:,3])
            if a > best[0]:
                best = (a, (w1, w2, w3, w4))
grid_auc, grid_w = best
print(f"\n4-way grid best: {dict(zip(names, grid_w))} -> {grid_auc:.5f}")

# choose the better of the two selection methods
hill_blend = current
if grid_auc >= roc_auc_score(y, hill_blend):
    final_name, final_w, final_oof_auc = 'grid_weights', grid_w, grid_auc
else:
    final_name, final_w, final_oof_auc = 'hill_climb_counts', {k: counts[k] for k in names}, roc_auc_score(y, hill_blend)
print(f"\nFINAL strategy: {final_name} | OOF AUC = {final_oof_auc:.5f}")
json.dump({'strategy': final_name, 'weights': dict(zip(names, final_w)),
           'oof_auc': float(final_oof_auc)},
          open(f'{WORK_DIR}/models/v2_final_strategy.json', 'w'), indent=2)

In [ ]:
# 10. Build final test predictions from the chosen recipe + submission CSV
TEST_M = np.column_stack([MEMBERS[k][1] for k in names])
if final_name == 'grid_weights':
    w = np.array(final_w)
    test_pred = w @ TEST_M.T
else:
    c = np.array([final_w[k] for k in names], dtype=float)
    test_pred = (c @ TEST_M.T) / c.sum()

print(f"test pred stats: mean={test_pred.mean():.5f} std={test_pred.std():.5f} "
      f"min={test_pred.min():.5f} max={test_pred.max():.5f}")

sample = pd.read_csv(f'{WORK_DIR}/input/sample_submission.csv')
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': test_pred})
submission = submission.set_index('SK_ID_CURR').loc[sample['SK_ID_CURR']].reset_index()

assert len(submission) == len(sample)
assert set(submission['SK_ID_CURR']) == set(sample['SK_ID_CURR'])
assert submission['TARGET'].between(0, 1).all()
assert not submission['TARGET'].isnull().any()
assert list(submission.columns) == ['SK_ID_CURR', 'TARGET']

import os
os.makedirs(f'{WORK_DIR}/submissions', exist_ok=True)
out = f'{WORK_DIR}/submissions/submission_v2_cv{final_oof_auc:.5f}.csv'
submission.to_csv(out, index=False)
submission.to_csv('submission_v2.csv', index=False)
print(f"saved: {out}")
print(submission.head())

In [ ]:
# 11. Summary
print("=" * 60)
print("NB3 v2 SUMMARY (all numbers = full-data 5-fold OOF)")
print("=" * 60)
print(f"LGB3 : {auc_lgb:.5f}")
print(f"XGB  : {auc_xgb:.5f}")
print(f"CAT  : {auc_cat:.5f}")
print(f"HGB  : {auc_hgb:.5f}")
print(f"FINAL ({final_name}): {final_oof_auc:.5f}")
print(f"\nOld pipeline verified OOF: 0.7587 -> new: {final_oof_auc:.5f}")